In [ ]:
! pip install psycopg2-binary

# 01 - Data loading and joining

## Syfte

Syftet med denna notebook är att läsa in relevanta tabeller från PostgreSQL, undersöka deras struktur och nycklar samt bygga ett gemensamt ML-dataset.

Varje tabell undersöks separat innan sammanslagning för att identifiera antal rader, unika nycklar och eventuella dubbletter. Tabeller som kan innehålla flera rader per order aggregeras innan de kopplas samman.

Målet är att skapa en slutlig ML-tabell där varje rad representerar exakt en order. Tabellen sparas som en artifact och används som input i nästa notebook.

In [ ]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from pathlib import Path

# Skapa sökvägen till mappen där projektets artifacts ska sparas
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

print("Artifacts directory:", artifacts_dir)

### Databasanslutning

I detta steg skapas en anslutning till PostgreSQL-databasen med SQLAlchemy. Anslutningen används senare för att läsa in de relevanta Olist-tabellerna direkt till pandas.

In [ ]:
# Ange anslutningsuppgifter till PostgreSQL-databasen
db_user = "postgres"
db_password = os.getenv("DB_PASSWORD")
db_host = "localhost"
db_port = "5432"
db_name = "olist"

# Skapa connection string för PostgreSQL
connection_string = (
    f"postgresql+psycopg2://{db_user}:{db_password}"
    f"@{db_host}:{db_port}/{db_name}"
)

# Skapa databasanslutningen
engine = create_engine(connection_string)

print("Connected to PostgreSQL successfully!")

### Tillgängliga tabeller i databasen

För att få en överblick över datakällan kontrollerar jag vilka tabeller som finns i PostgreSQL-databasen. Detta hjälper till att identifiera vilka tabeller som kan vara relevanta för den fortsatta analysen och sammanslagningen.

In [ ]:
# Visa vilka tabeller som finns i PostgreSQL-databasen
from sqlalchemy import inspect

# Hämta namnen på alla tabeller i databasen
inspector = inspect(engine)

table_names = inspector.get_table_names()

print("Tables in database:")
for table_name in table_names:
    print("-", table_name)

### Inspektion av `olist_orders`

`olist_orders` används som bastabell eftersom varje rad representerar en order. Innan andra tabeller kopplas till kontrolleras antal rader, nyckeln `order_id` och eventuella dubbletter.

In [ ]:
# Läs in ordertabellen
orders = pd.read_sql("SELECT * FROM olist_orders", engine)

print("Shape:", orders.shape)
print("Unique order_id:", orders["order_id"].nunique())
print("Duplicate order_id:", orders["order_id"].duplicated().sum())

orders.head()

### Resultat och tolkning

`olist_orders` innehåller 99 441 rader och lika många unika `order_id`. Det finns inga duplicerade `order_id`, vilket bekräftar att varje rad representerar en unik order.

Tabellen kan därför användas som bastabell för den slutliga ML-tabellen. Vid senare sammanslagningar behöver samma antal unika orders bevaras för att säkerställa att resultatet fortfarande innehåller exakt en rad per order.

### Inspektion av `olist_order_items`

`olist_order_items` innehåller information om produkterna i varje order. En order kan innehålla flera produkter och därför kan samma `order_id` förekomma på flera rader.

Tabellen behöver därför undersökas och senare aggregeras till ordernivå innan den kopplas till `olist_orders`.

In [ ]:
# Läs in order items
order_items = pd.read_sql("SELECT * FROM olist_order_items", engine)

print("Shape:", order_items.shape)
print("Unique order_id:", order_items["order_id"].nunique())
print("Duplicate order_id:", order_items["order_id"].duplicated().sum())

order_items.head()

### Resultat och tolkning

`olist_order_items` innehåller 112 650 rader men endast 98 666 unika `order_id`. Det finns 13 984 duplicerade förekomster av `order_id`, vilket visar att en order kan innehålla flera orderrader.

Tabellen kan därför inte kopplas direkt till `olist_orders`, eftersom det skulle skapa flera rader för vissa orders. Innan sammanslagningen behöver informationen aggregeras till ordernivå så att varje `order_id` förekommer exakt en gång.

# Not  :
Det här är anledningen till att vi använder groupby("order_id") senare.

### Aggregering av order items

Eftersom en order kan innehålla flera orderrader aggregeras `olist_order_items` till en rad per `order_id`.

Jag behåller antal orderrader samt totalt produktpris och total fraktkostnad. Dessa variabler sammanfattar orderns storlek och ekonomiska information utan att skapa flera rader per order.

In [ ]:
# Aggregera order items till en rad per order
items_agg = (
    order_items
    .groupby("order_id")
    .agg(
        total_items=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
    .reset_index()
)

print("Shape after aggregation:", items_agg.shape)
print("Unique order_id:", items_agg["order_id"].nunique())
print("Duplicate order_id:", items_agg["order_id"].duplicated().sum())

items_agg.head()

### Resultat och tolkning

Efter aggregeringen innehåller tabellen 98 666 rader och 98 666 unika `order_id`. Det finns inga duplicerade `order_id`.

`olist_order_items` har därmed omvandlats från orderradnivå till ordernivå. Variablerna `total_items`, `total_price` och `total_freight` kan nu kopplas till bastabellen utan att skapa flera rader per order.

# Not  :
olist_orders har 99 441 orders medan items_agg har 98 666. Det betyder att vissa orders saknar order-item-information. Vi tar inte bort dem nu; det kontrollerar vi efter vår left join.


### Inspektion av `olist_order_payments`

`olist_order_payments` innehåller betalningsinformation för orders. En order kan ha flera betalningsrader, exempelvis om flera betalningstransaktioner eller betalningsmetoder har registrerats.

Därför kontrolleras först antal rader, antal unika `order_id` och dubbletter innan tabellen används i en sammanslagning.

In [ ]:
# Läs in betalningsdata
payments = pd.read_sql("SELECT * FROM olist_order_payments", engine)

print("Shape:", payments.shape)
print("Unique order_id:", payments["order_id"].nunique())
print("Duplicate order_id:", payments["order_id"].duplicated().sum())

payments.head()

### Resultat och tolkning

`olist_order_payments` innehåller 103 886 rader men 99 440 unika `order_id`. Det finns 4 446 duplicerade förekomster av `order_id`, vilket visar att vissa orders har flera betalningsrader.

Tabellen kan därför inte kopplas direkt till `olist_orders`. Betalningsinformationen behöver först aggregeras till ordernivå för att den slutliga ML-tabellen ska behålla exakt en rad per order.

Det finns dessutom en order i bastabellen som saknar betalningsinformation. Denna order behålls tills vidare och undersöks efter sammanslagningen.

### Aggregering av betalningsdata

Eftersom en order kan ha flera betalningsrader aggregeras `olist_order_payments` till ordernivå innan sammanslagning.

Jag behåller det totala betalningsbeloppet, antal betalningsrader och högsta antal avbetalningar. På så sätt sammanfattas betalningsinformationen utan att skapa flera rader per order.

In [ ]:
# Aggregera betalningsdata till en rad per order
payments_agg = (
    payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        number_of_payments=("payment_sequential", "count"),
        max_installments=("payment_installments", "max")
    )
    .reset_index()
)

print("Shape after aggregation:", payments_agg.shape)
print("Unique order_id:", payments_agg["order_id"].nunique())
print("Duplicate order_id:", payments_agg["order_id"].duplicated().sum())

payments_agg.head()

### Resultat och tolkning

Efter aggregeringen innehåller betalningstabellen 99 440 rader och 99 440 unika `order_id`. Det finns inga duplicerade `order_id`.

Betalningsinformationen är därmed aggregerad till ordernivå och kan senare kopplas till bastabellen utan att skapa duplicerade orders. Den enda order som saknar betalningsinformation kommer att behållas genom en `left join` och undersökas efter sammanslagningen.

### Inspektion av `olist_customers`

`olist_customers` innehåller kundinformation kopplad till `customer_id`, bland annat postnummerprefix, stad och delstat.

Eftersom kundens geografiska information kan vara relevant senare i analysen kontrolleras först om `customer_id` är unik och om tabellen kan kopplas direkt till ordernivå utan att skapa dubbletter.

In [ ]:
# Läs in kunddata
customers = pd.read_sql("SELECT * FROM olist_customers", engine)

print("Shape:", customers.shape)
print("Unique customer_id:", customers["customer_id"].nunique())
print("Duplicate customer_id:", customers["customer_id"].duplicated().sum())

customers.head()

### Resultat och tolkning

`olist_customers` innehåller 99 441 rader och lika många unika `customer_id`. Det finns inga duplicerade `customer_id`.

Varje `customer_id` förekommer alltså exakt en gång i kundtabellen. Kundinformationen kan därför kopplas direkt till `olist_orders` via `customer_id` utan att skapa flera rader per order.

Variabler som `customer_zip_code_prefix`, `customer_city` och `customer_state` kan användas för att beskriva kundens geografiska information.

### Inspektion av `olist_sellers`

`olist_sellers` innehåller information om säljarna, bland annat postnummerprefix, stad och delstat.

Först kontrolleras om `seller_id` är unik i tabellen. Eftersom sellerinformationen kopplas till orders via `olist_order_items` behöver vi även senare kontrollera om en order kan innehålla produkter från flera olika sellers innan informationen aggregeras till ordernivå.

In [ ]:
# Läs in sellerdata
sellers = pd.read_sql("SELECT * FROM olist_sellers", engine)

print("Shape:", sellers.shape)
print("Unique seller_id:", sellers["seller_id"].nunique())
print("Duplicate seller_id:", sellers["seller_id"].duplicated().sum())

sellers.head()

### Resultat och tolkning

`olist_sellers` innehåller 3 095 rader och lika många unika `seller_id`. Det finns inga duplicerade `seller_id`, vilket innebär att varje seller förekommer exakt en gång i tabellen.

Sellerinformationen kan dock inte kopplas direkt till ordernivå enbart utifrån detta resultat. En order kan innehålla produkter från flera olika sellers, vilket behöver undersökas innan sellerinformationen kan representeras med en rad per order.

In [ ]:
# Kontrollera hur många unika sellers som förekommer per order
sellers_per_order = (
    order_items
    .groupby("order_id")["seller_id"]
    .nunique()
)

print("Number of orders:", sellers_per_order.shape[0])
print("Maximum sellers in one order:", sellers_per_order.max())
print("Orders with multiple sellers:", (sellers_per_order > 1).sum())

sellers_per_order.value_counts().sort_index()

### Resultat och tolkning

Analysen visar att 1 278 orders innehåller produkter från mer än en seller och att en enskild order kan innehålla upp till fem olika sellers.

Sellerinformationen kan därför inte kopplas direkt till ordertabellen, eftersom det skulle kunna skapa flera rader för samma order. Sellerinformationen behöver först representeras på ordernivå innan den används i den slutliga ML-tabellen.

För orders med flera sellers kommer en representativ seller senare att väljas på ett konsekvent sätt, samtidigt som antal unika sellers per order kan behållas som separat information.

### Inspektion av `olist_geolocation`

`olist_geolocation` innehåller geografisk information för postnummerprefix, bland annat latitud, longitud, stad och delstat.

Eftersom samma postnummerprefix kan förekomma på flera rader behöver tabellen undersökas innan den kopplas till kund- och sellerdata. Om ett postnummerprefix förekommer flera gånger behöver koordinaterna först aggregeras till en representativ position per postnummerprefix.

In [ ]:
# Läs in geografidata
geolocation = pd.read_sql("SELECT * FROM olist_geolocation", engine)

print("Shape:", geolocation.shape)
print(
    "Unique zip code prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)
print(
    "Duplicate zip code prefixes:",
    geolocation["geolocation_zip_code_prefix"].duplicated().sum()
)

geolocation.head()

### Resultat och tolkning

`olist_geolocation` innehåller 1 000 163 rader men endast 19 015 unika postnummerprefix. Det finns därför många observationer för samma `geolocation_zip_code_prefix`.

Tabellen kan inte kopplas direkt till kund- eller sellerdata eftersom det skulle kunna skapa ett stort antal duplicerade rader. Geografidata behöver först aggregeras till en rad per postnummerprefix.

För varje postnummerprefix används medelvärdet av latitud och longitud som en representativ geografisk position.

In [ ]:
# Aggregera geografidata till en rad per postnummerprefix
geolocation_agg = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "mean"),
        longitude=("geolocation_lng", "mean")
    )
    .reset_index()
)

print("Shape after aggregation:", geolocation_agg.shape)
print(
    "Unique zip code prefixes:",
    geolocation_agg["geolocation_zip_code_prefix"].nunique()
)
print(
    "Duplicate zip code prefixes:",
    geolocation_agg["geolocation_zip_code_prefix"].duplicated().sum()
)

geolocation_agg.head()

### Resultat efter aggregering

Efter aggregeringen innehåller geografitabellen 19 015 rader och 19 015 unika postnummerprefix. Det finns inga duplicerade `geolocation_zip_code_prefix`.

Varje postnummerprefix representeras nu av en geografisk position baserad på medelvärdet av latitud och longitud. Tabellen kan därför senare användas för att koppla koordinater till kundens och sellerns postnummerprefix utan att skapa duplicerade orderrader.

# NOT 
Den aggregerade geografitabellen kan senare användas för att beräkna avståndet mellan kund och seller. Jag väntar med att skapa denna variabel tills alla relevanta tabeller har inspekterats och sellerinformationen har representerats korrekt på ordernivå.

### Inspektion av `olist_products`

`olist_products` innehåller information om produkter, exempelvis produktkategori samt olika mått och dimensioner.

Eftersom produkter kopplas till orders via `olist_order_items` undersöks först tabellens storlek, nyckel och eventuella dubbletter innan det beslutas om produktinformationen ska användas i den slutliga ML-tabellen.

In [ ]:
# Läs in produktdata
products = pd.read_sql("SELECT * FROM olist_products", engine)

print("Shape:", products.shape)
print("Unique product_id:", products["product_id"].nunique())
print("Duplicate product_id:", products["product_id"].duplicated().sum())

products.head()

### Resultat och beslut

`olist_products` innehåller 32 951 rader och varje `product_id` är unik. Tabellen är därför korrekt strukturerad på produktnivå.

En order kan däremot innehålla flera produkter via `olist_order_items`. Produktinformationen kan därför inte kopplas direkt till ordertabellen utan ytterligare aggregering på ordernivå.

I denna version av ML-tabellen används redan ordernivåinformation om antal produkter, pris och frakt från `olist_order_items`. Produktattribut från `olist_products` inkluderas därför inte i den slutliga ML-tabellen.

### Inspektion av `olist_order_reviews`

`olist_order_reviews` innehåller kundrecensioner kopplade till orders, exempelvis betyg, kommentarer och tidpunkter för recensionen.

Tabellen undersöks separat för att kontrollera antal rader, unika order-ID:n och eventuella dubbletter. Eftersom recensioner skapas efter att en order har genomförts behöver informationen även bedömas ur ett perspektiv av target leakage innan den eventuellt används i ML-tabellen.

In [ ]:
# Läs in recensionsdata
reviews = pd.read_sql("SELECT * FROM olist_order_reviews", engine)

print("Shape:", reviews.shape)
print("Unique review_id:", reviews["review_id"].nunique())
print("Duplicate review_id:", reviews["review_id"].duplicated().sum())
print("Unique order_id:", reviews["order_id"].nunique())
print("Duplicate order_id:", reviews["order_id"].duplicated().sum())

reviews.head()

### Resultat och beslut

`olist_order_reviews` innehåller 98 410 unika recensioner. `review_id` är unik, men `order_id` är inte unik eftersom vissa orders har fler än en recension.

Recensionsinformationen skapas efter att ordern har genomförts och är därför inte tillgänglig vid den tidpunkt då en prediktion om försenad leverans ska göras. Att använda exempelvis `review_score` eller recensionskommentarer som features skulle innebära target leakage.

`olist_order_reviews` inkluderas därför inte i den slutliga ML-tabellen.

### Inspektion av `product_category_name_translation`

`product_category_name_translation` är en uppslagstabell som översätter produktkategorier från portugisiska till engelska.

Tabellen undersöks för att kontrollera antal kategorier, unikhet och eventuella dubbletter. Eftersom produktkategorier inte används i den slutliga ML-tabellen används inte heller översättningstabellen i den fortsatta sammanslagningen.

In [ ]:
# Läs in översättningstabellen för produktkategorier
category_translation = pd.read_sql(
    "SELECT * FROM product_category_name_translation",
    engine
)

print("Shape:", category_translation.shape)
print(
    "Unique product_category_name:",
    category_translation["product_category_name"].nunique()
)
print(
    "Duplicate product_category_name:",
    category_translation["product_category_name"].duplicated().sum()
)

category_translation.head()

### Resultat och beslut

`product_category_name_translation` innehåller 71 produktkategorier. Varje `product_category_name` är unik och det finns inga dubbletter.

Tabellen fungerar endast som en översättning av produktkategorier. Eftersom produktkategorier från `olist_products` inte inkluderas i den slutliga ML-tabellen behövs inte heller denna översättningstabell i den fortsatta sammanslagningen.

In [ ]:
# Summera produktvärdet per seller och order
seller_order_value = (
    order_items
    .groupby(["order_id", "seller_id"], as_index=False)
    .agg(seller_order_value=("price", "sum"))
)

# Beräkna antal unika sellers per order
seller_order_value["seller_count"] = (
    seller_order_value
    .groupby("order_id")["seller_id"]
    .transform("nunique")
)

# Välj den seller som har högst sammanlagt produktvärde i varje order
primary_seller = (
    seller_order_value
    .sort_values(
        ["order_id", "seller_order_value", "seller_id"],
        ascending=[True, False, True]
    )
    .drop_duplicates("order_id")
    .rename(columns={"seller_id": "primary_seller_id"})
    [["order_id", "primary_seller_id", "seller_count"]]
)

print("Shape:", primary_seller.shape)
print("Unique order_id:", primary_seller["order_id"].nunique())
print("Duplicate order_id:", primary_seller["order_id"].duplicated().sum())
print("Maximum seller_count:", primary_seller["seller_count"].max())

primary_seller.head()

### Resultat och tolkning

Efter bearbetningen innehåller `primary_seller` 98 666 orders och varje `order_id` är unik. Det finns inga duplicerade orderrader och det maximala antalet sellers i en order är fem.

För orders med flera sellers representeras ordern av den seller som står för det högsta sammanlagda produktvärdet. Vid lika produktvärde används `seller_id` som ett konsekvent sätt att avgöra vilken seller som väljs.

Variabeln `seller_count` behålls för att även representera hur många unika sellers som ingår i ordern. På detta sätt kan sellerinformation senare kopplas till ML-tabellen utan att bryta kravet på en rad per order.

In [ ]:
# Koppla den representativa sellern till geografisk sellerinformation
primary_seller = primary_seller.merge(
    sellers[
        ["seller_id", "seller_zip_code_prefix", "seller_state"]
    ].rename(columns={"seller_id": "primary_seller_id"}),
    on="primary_seller_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", primary_seller.shape)
print("Unique order_id:", primary_seller["order_id"].nunique())
print("Duplicate order_id:", primary_seller["order_id"].duplicated().sum())

print("\nMissing seller information:")
print(
    primary_seller[
        ["seller_zip_code_prefix", "seller_state"]
    ].isna().sum()
)

primary_seller.head()

### Resultat efter koppling till sellerdata

Efter kopplingen finns fortfarande 98 666 unika orders och inga duplicerade `order_id`. Det finns inga saknade värden i `seller_zip_code_prefix` eller `seller_state` för de orders som har sellerinformation.

Sellerinformationen är därmed representerad på ordernivå och kan användas i den fortsatta sammanslagningen utan att skapa flera rader per order.

In [ ]:
# Förbered geografiska koordinater för kunder
customer_geo = geolocation_agg.rename(
    columns={
        "geolocation_zip_code_prefix": "customer_zip_code_prefix",
        "latitude": "customer_latitude",
        "longitude": "customer_longitude"
    }
)

# Förbered geografiska koordinater för sellers
seller_geo = geolocation_agg.rename(
    columns={
        "geolocation_zip_code_prefix": "seller_zip_code_prefix",
        "latitude": "seller_latitude",
        "longitude": "seller_longitude"
    }
)

print("Customer geography shape:", customer_geo.shape)
print("Seller geography shape:", seller_geo.shape)

print(
    "Duplicate customer zip prefixes:",
    customer_geo["customer_zip_code_prefix"].duplicated().sum()
)

print(
    "Duplicate seller zip prefixes:",
    seller_geo["seller_zip_code_prefix"].duplicated().sum()
)

customer_geo.head()

### Resultat för geografiska uppslagstabeller

Både kund- och sellergeografin innehåller 19 015 unika postnummerprefix och inga dubbletter.

Koordinaterna har fått separata kolumnnamn för kunder och sellers. De kan därför senare kopplas till respektive postnummerprefix utan att skapa duplicerade orderrader eller blanda ihop kundens och sellerns geografiska position.

##  ML-tabell

`olist_orders` används som bastabell eftersom varje rad representerar en unik order. Övrig information kopplas till denna tabell med left joins för att bevara samtliga orders.

Tabeller med flera rader per order har redan aggregerats till ordernivå innan sammanslagningen. Efter varje steg kontrolleras antal rader och unika `order_id` för att säkerställa att inga duplicerade orderrader skapas.

In [ ]:
# Starta från ordertabellen och koppla aggregerad order-item-information
ml_table = orders.merge(
    items_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Shape:", ml_table.shape)
print("Unique order_id:", ml_table["order_id"].nunique())
print("Duplicate order_id:", ml_table["order_id"].duplicated().sum())

print("\nOrders without item information:")
print(ml_table["total_items"].isna().sum())

ml_table.head()

### Resultat efter koppling av order-item-information

Efter sammanslagningen finns fortfarande 99 441 unika orders och inga duplicerade `order_id`.

775 orders saknar aggregerad order-item-information. Dessa orders behålls i datasetet eftersom `olist_orders` används som bastabell och sammanslagningen görs med en left join. Saknade värden dokumenteras i detta steg och hanteras inte genom imputering i denna notebook.

In [ ]:
# Koppla aggregerad betalningsinformation till ML-tabellen
ml_table = ml_table.merge(
    payments_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Shape:", ml_table.shape)
print("Unique order_id:", ml_table["order_id"].nunique())
print("Duplicate order_id:", ml_table["order_id"].duplicated().sum())

print("\nOrders without payment information:")
print(ml_table["total_payment_value"].isna().sum())

ml_table.head()

### Resultat efter koppling av betalningsinformation

Efter sammanslagningen finns fortfarande 99 441 unika orders och inga duplicerade `order_id`.

Endast en order saknar betalningsinformation. Ordern behålls i datasetet och det saknade värdet dokumenteras i detta steg. Ingen imputering görs i denna notebook eftersom hantering av saknade värden hör till den senare feature engineering-processen.

In [ ]:
# Koppla kundens geografiska information till ML-tabellen
ml_table = ml_table.merge(
    customers[
        ["customer_id", "customer_zip_code_prefix", "customer_state"]
    ],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Shape:", ml_table.shape)
print("Unique order_id:", ml_table["order_id"].nunique())
print("Duplicate order_id:", ml_table["order_id"].duplicated().sum())

print("\nMissing customer information:")
print(
    ml_table[
        ["customer_zip_code_prefix", "customer_state"]
    ].isna().sum()
)

ml_table.head()

### Resultat efter koppling av kundinformation

Efter sammanslagningen finns fortfarande 99 441 unika orders och inga duplicerade `order_id`.

Samtliga orders har tillgänglig information om kundens postnummerprefix och delstat. Kundinformationen kan därför användas i den fortsatta geografiska analysen utan ytterligare hantering i detta steg.

In [ ]:
# Koppla kundens geografiska koordinater till ML-tabellen
ml_table = ml_table.merge(
    customer_geo,
    on="customer_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

print("Shape:", ml_table.shape)
print("Unique order_id:", ml_table["order_id"].nunique())
print("Duplicate order_id:", ml_table["order_id"].duplicated().sum())

print("\nMissing customer coordinates:")
print(
    ml_table[
        ["customer_latitude", "customer_longitude"]
    ].isna().sum()
)

ml_table.head()

### Resultat efter koppling av kundkoordinater

Efter kopplingen finns fortfarande 99 441 unika orders och inga duplicerade `order_id`.

För 278 orders saknas geografiska koordinater för kunden, trots att `customer_zip_code_prefix` finns tillgängligt. Detta innebär att vissa kunders postnummerprefix saknar motsvarande geografisk information i den aggregerade geolocation-tabellen.

Dessa orders behålls i datasetet. De saknade koordinaterna dokumenteras i detta steg och hanteras senare i feature engineering-processen.

In [ ]:
# Koppla representativ sellerinformation till ML-tabellen
ml_table = ml_table.merge(
    primary_seller,
    on="order_id",
    how="left",
    validate="one_to_one"
)

print("Shape:", ml_table.shape)
print("Unique order_id:", ml_table["order_id"].nunique())
print("Duplicate order_id:", ml_table["order_id"].duplicated().sum())

print("\nMissing seller information:")
print(
    ml_table[
        [
            "primary_seller_id",
            "seller_count",
            "seller_zip_code_prefix",
            "seller_state"
        ]
    ].isna().sum()
)

ml_table.head()

### Resultat efter koppling av sellerinformation

Efter sammanslagningen finns fortfarande 99 441 unika orders och inga duplicerade `order_id`.

För 775 orders saknas information om representativ seller, antal sellers, sellerns postnummerprefix och delstat. Dessa orders motsvarar de orders som saknar order-item-information och därför inte kan kopplas till någon seller.

Orders utan sellerinformation behålls i datasetet. De saknade värdena dokumenteras och hanteras senare i feature engineering-processen.

In [ ]:
# Koppla sellerns geografiska koordinater till ML-tabellen
ml_table = ml_table.merge(
    seller_geo,
    on="seller_zip_code_prefix",
    how="left",
    validate="many_to_one"
)

print("Shape:", ml_table.shape)
print("Unique order_id:", ml_table["order_id"].nunique())
print("Duplicate order_id:", ml_table["order_id"].duplicated().sum())

print("\nMissing seller coordinates:")
print(
    ml_table[
        ["seller_latitude", "seller_longitude"]
    ].isna().sum()
)

ml_table.head()

### Resultat efter koppling av sellerkoordinater

Efter kopplingen finns fortfarande 99 441 unika orders och inga duplicerade `order_id`.

För 991 orders saknas geografiska koordinater för sellern. Av dessa saknar 775 orders sellerinformation helt eftersom de inte har någon order-item-information. För övriga orders finns sellerinformation, men sellerns postnummerprefix saknar motsvarande koordinater i den aggregerade geolocation-tabellen.

Dessa orders behålls i datasetet. De saknade geografiska värdena dokumenteras i detta steg och hanteras senare i feature engineering-processen.

## Geografiskt avstånd mellan kund och seller

För att representera det geografiska avståndet mellan kunden och den representativa sellern beräknas fågelvägsavståndet med Haversine-formeln.

Beräkningen använder de aggregerade koordinaterna för kundens och sellerns postnummerprefix. Avståndet är därför en geografisk approximation i kilometer och representerar inte den faktiska transport- eller körsträckan.

In [ ]:
# Beräkna geografiskt avstånd i kilometer med Haversine-formeln
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(
        np.radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * 6371 * np.arcsin(np.sqrt(a))


ml_table["distance_km"] = haversine_km(
    ml_table["customer_latitude"],
    ml_table["customer_longitude"],
    ml_table["seller_latitude"],
    ml_table["seller_longitude"]
)

print(ml_table["distance_km"].describe())

print("\nMissing distance_km:")
print(ml_table["distance_km"].isna().sum())

### Resultat för geografiskt avstånd

Geografiskt avstånd kunde beräknas för 98 177 orders. Medianavståndet är cirka 434 km och medelvärdet cirka 602 km.

För 1 264 orders saknas `distance_km` eftersom koordinater saknas för kunden, sellern eller båda. Dessa orders behålls och de saknade värdena hanteras senare i feature engineering-processen.

Avståndsvariabelns fördelning och eventuella extrema värden analyseras mer detaljerat i EDA-notebooken.

In [ ]:
# Ta bort tillfälliga kolumner som användes för geografiberäkningen
columns_to_drop = [
    "primary_seller_id",
    "customer_latitude",
    "customer_longitude",
    "seller_latitude",
    "seller_longitude"
]

ml_table = ml_table.drop(columns=columns_to_drop)

print("Shape:", ml_table.shape)
print("Unique order_id:", ml_table["order_id"].nunique())
print("Duplicate order_id:", ml_table["order_id"].duplicated().sum())

ml_table.head()

In [ ]:
# Slutlig kontroll av ML-tabellen före lagring
print("Final shape:", ml_table.shape)
print("Unique order_id:", ml_table["order_id"].nunique())
print("Duplicate order_id:", ml_table["order_id"].duplicated().sum())

print("\nMissing values:")
print(
    ml_table
    .isna()
    .sum()
    .sort_values(ascending=False)
)

### Slutlig kontroll av saknade värden

Den slutliga ML-tabellen innehåller 99 441 unika orders och inga duplicerade `order_id`.

Saknade värden finns främst i leveransdatum, geografiskt avstånd samt order-item- och sellerinformation. De 775 orders som saknar item-information saknar även motsvarande sellerinformation. `distance_km` saknas för 1 264 orders där geografiska koordinater inte finns tillgängliga för kunden, sellern eller båda.

En order saknar betalningsinformation. Kundens postnummerprefix och delstat är däremot kompletta för samtliga orders.

Inga rader tas bort och ingen imputering görs i denna notebook. Saknade värden behålls och hanteras i senare steg där deras betydelse kan analyseras och lämplig preprocessing kan bestämmas.

In [ ]:
# Undersök orderstatus för orders som saknar order-item-information
missing_items_status = (
    ml_table.loc[
        ml_table["total_items"].isna(),
        "order_status"
    ]
    .value_counts()
)

print("Orders without item information:", ml_table["total_items"].isna().sum())
print("\nOrder status:")
print(missing_items_status)

### Tolkning av orders utan item-information

De 775 orders som saknar order-item-information består främst av orders med status `unavailable` (603) och `canceled` (164). Endast ett fåtal har status `created`, `invoiced` eller `shipped`.

Detta visar att den saknade item- och sellerinformationen till stor del är kopplad till orderns status och därför inte bör betraktas som helt slumpmässig. Orders behålls i ML-tabellen och ingen imputering görs i detta steg.

In [ ]:
# Undersök ordern som saknar betalningsinformation
missing_payment_orders = ml_table.loc[
    ml_table["total_payment_value"].isna(),
    ["order_id", "order_status", "order_purchase_timestamp"]
]

print("Orders without payment information:", len(missing_payment_orders))
missing_payment_orders

### Resultat för saknad betalningsinformation

Endast en order saknar betalningsinformation. Ordern har status `delivered`, vilket visar att avsaknaden av betalningsdata inte kan förklaras av att ordern exempelvis har avbrutits eller varit otillgänglig.

Ordern behålls i ML-tabellen och inget betalningsvärde imputeras i denna notebook. Det saknade värdet hanteras senare i preprocessing- och feature engineering-steget.

In [ ]:
# Spara den slutliga ML-tabellen som artifact
artifact_path = artifacts_dir / "ml_table.csv"

ml_table.to_csv(
    artifact_path,
    index=False
)

print("Saved:", artifact_path)
print("File exists:", artifact_path.exists())
print("Final shape:", ml_table.shape)

## SISTA RESULTAT 

I denna notebook har de relevanta tabellerna i PostgreSQL undersökts separat med fokus på antal rader, nycklar, dubbletter och vilken nivå varje tabell representerar.

`olist_orders` användes som bastabell eftersom varje rad representerar en unik order. `olist_order_items` och `olist_order_payments` innehåller flera rader per order och aggregerades därför till ordernivå innan sammanslagningen.

Kundinformation kopplades direkt via `customer_id`. Eftersom en order kan innehålla flera sellers valdes en representativ seller baserat på högst sammanlagt produktvärde inom ordern, samtidigt som antalet unika sellers per order sparades i `seller_count`.

Geolocation-data aggregerades per postnummerprefix. Kundens och sellerns geografiska position användes därefter för att beräkna ett approximativt fågelvägsavstånd i kilometer (`distance_km`). Kund- och sellerstate samt respektive postnummerprefix behölls för senare geografisk analys.

Övriga databastabeller undersöktes också. Produktdata och kategoriöversättningar inkluderades inte eftersom de inte behövdes för de valda orderbaserade variablerna. Reviewdata exkluderades eftersom den skapas efter ordern och därför kan innebära framtidsinformation vid prediktion.

Den slutliga ML-tabellen innehåller 99 441 rader och 20 kolumner, med exakt en rad per order och inga duplicerade `order_id`. Saknade värden har dokumenterats men inte imputerats eller tagits bort i denna notebook.

Den färdiga tabellen sparades som `artifacts/ml_table.csv` och används som input till nästa notebook.